In [0]:
select * from interview_catalog.source.employee

In [0]:
select max(salary) from interview_catalog.source.employee;
select salary from interview_catalog.source.employee order by salary desc limit 1;

-- 2nd highest

select max(salary) from interview_catalog.source.employee where salary < (select max(salary) from interview_catalog.source.employee);
select salary from interview_catalog.source.employee order by salary desc limit 1 offset 1;

--n th highest

select salary from(
    select salary, dense_rank() over(order by salary desc) as rn from interview_catalog.source.employee
) where rn = 4



In [0]:
select max(salary) from interview_catalog.source.employee;
select salary from interview_catalog.source.employee order by salary desc limit 1;

-- 2nd highest

select max(salary) from interview_catalog.source.employee where salary < (select max(salary) from interview_catalog.source.employee);
select salary from interview_catalog.source.employee order by salary desc limit 1 offset 1;

--n th highest

select salary from(
    select salary, dense_rank() over(order by salary desc) as rn from interview_catalog.source.employee
) where rn = 4



In [0]:
-- ========================================
-- ALTERNATIVE METHODS FOR FINDING SALARIES
-- ========================================

-- METHOD 1: Using ROW_NUMBER() (excludes duplicates)
-- Different from DENSE_RANK: If two employees have same salary, ROW_NUMBER assigns different ranks
SELECT salary 
FROM (
    SELECT salary, ROW_NUMBER() OVER(ORDER BY salary DESC) as rn 
    FROM interview_catalog.source.employee
) 
WHERE rn = 2;

-- METHOD 2: Using RANK() (skips ranks after duplicates)
-- If 2 employees have highest salary, next rank is 3 (not 2)
SELECT salary 
FROM (
    SELECT salary, RANK() OVER(ORDER BY salary DESC) as rnk 
    FROM interview_catalog.source.employee
) 
WHERE rnk = 2;

-- METHOD 3: Using QUALIFY clause (cleaner syntax)
-- QUALIFY filters window function results directly
SELECT DISTINCT salary
FROM interview_catalog.source.employee
QUALIFY DENSE_RANK() OVER(ORDER BY salary DESC) = 2;

-- METHOD 4: Self-join approach (classic method)
-- Count how many distinct salaries are higher
SELECT DISTINCT e1.salary
FROM interview_catalog.source.employee e1
WHERE 1 = (
    SELECT COUNT(DISTINCT e2.salary)
    FROM interview_catalog.source.employee e2
    WHERE e2.salary > e1.salary
);

-- METHOD 5: Using correlated subquery
-- Find salary where exactly N-1 salaries are greater
SELECT DISTINCT salary
FROM interview_catalog.source.employee e1
WHERE (
    SELECT COUNT(DISTINCT salary)
    FROM interview_catalog.source.employee e2
    WHERE e2.salary > e1.salary
) = 1;  -- Change to 1 for 2nd highest, 2 for 3rd highest, etc.

-- METHOD 6: Using DISTINCT with window function (for unique salaries only)
SELECT salary
FROM (
    SELECT DISTINCT salary, DENSE_RANK() OVER(ORDER BY salary DESC) as rn
    FROM interview_catalog.source.employee
)
WHERE rn = 2;

-- METHOD 7: Using ARRAY_AGG and array indexing
-- Collect all distinct salaries into array, then access by index
SELECT sorted_salaries[1] as nth_highest_salary
FROM (
    SELECT ARRAY_SORT(COLLECT_SET(salary), (left, right) -> CASE WHEN left > right THEN -1 ELSE 1 END) as sorted_salaries
    FROM interview_catalog.source.employee
);

-- METHOD 8: Using LAG window function (compare with previous row)
SELECT salary
FROM (
    SELECT DISTINCT salary,
           LAG(salary, 1) OVER(ORDER BY salary DESC) as prev_salary,
           ROW_NUMBER() OVER(ORDER BY salary DESC) as rn
    FROM interview_catalog.source.employee
)
WHERE rn = 2;

-- METHOD 9: Top N salaries with their counts
-- Useful to see distribution of top salaries
SELECT salary, COUNT(*) as employee_count
FROM interview_catalog.source.employee
GROUP BY salary
ORDER BY salary DESC
LIMIT 5;

In [0]:
-- ============================================
-- SQL PRACTICE SCENARIOS - Employee Dataset
-- ============================================

-- SCENARIO 1: Basic Aggregation
-- Find the total number of employees in each department and their average salary
SELECT department, 
       COUNT(*) as total_employees,
       AVG(salary) as avg_salary,
       MIN(salary) as min_salary,
       MAX(salary) as max_salary
FROM interview_catalog.source.employee
GROUP BY department
ORDER BY avg_salary DESC;


-- SCENARIO 3: Find Employees Earning More Than Their Manager
SELECT e.emp_name as employee,
       e.salary as employee_salary,
       m.emp_name as manager,
       m.salary as manager_salary,
       (e.salary - m.salary) as salary_difference
FROM interview_catalog.source.employee e
INNER JOIN interview_catalog.source.employee m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;

-- SCENARIO 4: Department Salary Analysis with Window Functions
-- Show each employee's salary rank within their department
SELECT emp_name,
       department,
       job_title,
       salary,
       RANK() OVER(PARTITION BY department ORDER BY salary DESC) as dept_salary_rank,
       DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as dept_dense_rank,
       ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC) as row_num
FROM interview_catalog.source.employee
ORDER BY department, salary DESC;

-- SCENARIO 5: Salary Comparison with Department Average
-- Show employees whose salary is above department average
SELECT e.emp_name,
       e.department,
       e.salary,
       ROUND(AVG(e2.salary), 2) as dept_avg_salary,
       ROUND(e.salary - AVG(e2.salary), 2) as difference_from_avg,
       ROUND((e.salary - AVG(e2.salary)) / AVG(e2.salary) * 100, 2) as pct_above_avg
FROM interview_catalog.source.employee e
INNER JOIN interview_catalog.source.employee e2 ON e.department = e2.department
GROUP BY e.emp_name, e.department, e.salary
HAVING e.salary > AVG(e2.salary)
ORDER BY pct_above_avg DESC;

-- SCENARIO 6: Tenure Analysis
-- Calculate years of service and categorize employees
SELECT emp_name,
       department,
       hire_date,
       DATEDIFF(CURRENT_DATE(), hire_date) as days_employed,
       ROUND(DATEDIFF(CURRENT_DATE(), hire_date) / 365.25, 1) as years_employed,
       CASE 
           WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 > 6 THEN 'Veteran'
           WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 > 4 THEN 'Experienced'
           WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 > 2 THEN 'Mid-Level'
           ELSE 'New Hire'
       END as tenure_category
FROM interview_catalog.source.employee
ORDER BY years_employed DESC;

-- SCENARIO 7: Top 3 Highest Paid Employees per Department
SELECT *
FROM (
    SELECT emp_name,
           department,
           job_title,
           salary,
           DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as salary_rank
    FROM interview_catalog.source.employee
)
WHERE salary_rank <= 3
ORDER BY department, salary_rank;

-- SCENARIO 8: Duplicate Detection
-- Find employees with same name (potential duplicates)
SELECT emp_name, COUNT(*) as count
FROM interview_catalog.source.employee
GROUP BY emp_name
HAVING COUNT(*) > 1;

-- SCENARIO 9: Salary Gaps Analysis
-- Find salary difference between consecutive employees within department
SELECT emp_name,
       department,
       salary,
       LAG(salary) OVER(PARTITION BY department ORDER BY salary DESC) as next_higher_salary,
       LAG(salary) OVER(PARTITION BY department ORDER BY salary DESC) - salary as salary_gap
FROM interview_catalog.source.employee
ORDER BY department, salary DESC;

-- SCENARIO 10: Department Head Count by Job Title
-- Pivot-like view of employee distribution
SELECT department,
       COUNT(*) as total_employees,
       COUNT(CASE WHEN job_title LIKE '%Manager%' THEN 1 END) as managers,
       COUNT(CASE WHEN job_title LIKE '%Engineer%' THEN 1 END) as engineers,
       COUNT(CASE WHEN job_title LIKE '%Analyst%' THEN 1 END) as analysts,
       COUNT(CASE WHEN job_title LIKE '%Scientist%' THEN 1 END) as scientists
FROM interview_catalog.source.employee
GROUP BY department;

-- SCENARIO 11: Running Total of Salaries by Hire Date
-- Calculate cumulative salary expenditure over time
SELECT emp_name,
       department,
       hire_date,
       salary,
       SUM(salary) OVER(ORDER BY hire_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total_salary,
       COUNT(*) OVER(ORDER BY hire_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as cumulative_headcount
FROM interview_catalog.source.employee
ORDER BY hire_date;

-- SCENARIO 12: Find Employees Hired in Same Month
-- Group employees by hire month and year
SELECT YEAR(hire_date) as hire_year,
       MONTH(hire_date) as hire_month,
       CONCAT(YEAR(hire_date), '-', LPAD(MONTH(hire_date), 2, '0')) as hire_month_year,
       COUNT(*) as employees_hired,
       GROUP_CONCAT(emp_name) as employee_names
FROM interview_catalog.source.employee
GROUP BY hire_year, hire_month
HAVING COUNT(*) > 1
ORDER BY hire_year DESC, hire_month DESC;

-- SCENARIO 13: Salary Percentile Analysis
-- Find employees in top 25% salary range
SELECT emp_name,
       department,
       salary,
       PERCENT_RANK() OVER(ORDER BY salary) as salary_percentile,
       NTILE(4) OVER(ORDER BY salary) as salary_quartile
FROM interview_catalog.source.employee
QUALIFY salary_quartile = 4  -- Top quartile
ORDER BY salary DESC;

-- SCENARIO 14: Manager's Team Statistics
-- Show each manager's team size and salary budget
SELECT m.emp_id as manager_id,
       m.emp_name as manager_name,
       m.department,
       COUNT(e.emp_id) as team_size,
       SUM(e.salary) as total_team_salary,
       AVG(e.salary) as avg_team_salary,
       m.salary as manager_salary
FROM interview_catalog.source.employee m
INNER JOIN interview_catalog.source.employee e ON m.emp_id = e.manager_id
GROUP BY m.emp_id, m.emp_name, m.department, m.salary
ORDER BY team_size DESC;

-- SCENARIO 15: Complex Query - Department Performance Report
-- Comprehensive department analysis
WITH dept_stats AS (
    SELECT department,
           COUNT(*) as employee_count,
           SUM(salary) as total_salary,
           AVG(salary) as avg_salary,
           MAX(salary) as max_salary,
           MIN(salary) as min_salary
    FROM interview_catalog.source.employee
    GROUP BY department
),
manager_count AS (
    SELECT department,
           COUNT(DISTINCT manager_id) as unique_managers
    FROM interview_catalog.source.employee
    WHERE manager_id IS NOT NULL
    GROUP BY department
)
SELECT d.department,
       d.employee_count,
       d.total_salary,
       ROUND(d.avg_salary, 2) as avg_salary,
       d.max_salary,
       d.min_salary,
       (d.max_salary - d.min_salary) as salary_range,
       COALESCE(m.unique_managers, 0) as managers_count,
       ROUND(d.total_salary / d.employee_count, 2) as per_employee_cost
FROM dept_stats d
LEFT JOIN manager_count m ON d.department = m.department
ORDER BY d.total_salary DESC;

In [0]:
-- SCENARIO 2: Employee Hierarchy (Self-Join)
-- List employees with their manager names
SELECT e.emp_id,
       e.emp_name as employee_name,
       e.job_title,
       e.salary,
       m.emp_name as manager_name,
       m.job_title as manager_title
FROM interview_catalog.source.employee e
LEFT JOIN interview_catalog.source.employee m ON e.manager_id = m.emp_id
ORDER BY e.department, e.salary DESC;


In [0]:
-- SCENARIO 3: Find Employees Earning More Than Their Manager
SELECT e.emp_name as employee,
       e.salary as employee_salary,
       m.emp_name as manager,
       m.salary as manager_salary,
       (e.salary - m.salary) as salary_difference
FROM interview_catalog.source.employee e
INNER JOIN interview_catalog.source.employee m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;


In [0]:
-- SCENARIO 4: Department Salary Analysis with Window Functions
-- Show each employee's salary rank within their department

select * , dense_rank() over ( partition by department order by salary desc) as salary_rank from interview_catalog.source.employee

In [0]:
-- SCENARIO 5: Salary Comparison with Department Average
-- Show employees whose salary is above department average

select emp_name, salary, department from(
    select *, avg(salary) over (partition by department) as avg_salary from  interview_catalog.source.employee
) where salary > avg_salary


In [0]:
select emp_name, salary, department from(
    select *, dense_rank() over (partition by department order by salary desc)  as rnk from  interview_catalog.source.employee
) where rnk <=3

In [0]:
-- SCENARIO 12: Find Employees Hired in Same Month
-- Group employees by hire month and year
SELECT YEAR(hire_date) as hire_year,
       MONTH(hire_date) as hire_month,
       CONCAT(YEAR(hire_date), '-', LPAD(MONTH(hire_date), 2, '0')) as hire_month_year,
       COUNT(*) as employees_hired
    --    GROUP_CONCAT(emp_name) as employee_names
FROM interview_catalog.source.employee
GROUP BY hire_year, hire_month
HAVING COUNT(*) > 1
ORDER BY hire_year DESC, hire_month DESC;